In [ ]:
 !nvidia-smi

Sun Nov  8 15:38:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu pymupdf

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "ibm-granite/granite-3.3-2b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cuda"
)
print("Granite loaded!")

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Granite loaded!


In [ ]:
def ask(question, max_new_tokens=200):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("In two sentences, what is retrieval-augmented generation?"))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Retrieval-augmented generation is a method in natural language processing that combines the strengths of retrieval and generation models. It first retrieves relevant information from a large corpus of text using a retrieval model, then uses a generation model to create a more contextually accurate and coherent response.


In [ ]:
import os, fitz

path = next(p for p in ["/content/testpaper.pdf", "/testpaper.pdf"] if os.path.exists(p))
doc = fitz.open(path)
pages = [page.get_text() for page in doc]
text = "\n".join(pages)

print("Found:", path)
print("Pages:", len(pages))
print("Characters:", len(text))
print("---")
print(text[:500])

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Found: /testpaper.pdf
Pages: 35
Characters: 89386
---
1 
 
A Multimodal Engineering GraphRAG Framework for Post-Wind-Event 
Building Assessment and Recovery Inference 
Seyedeh Fatemeh Mirfakhar1, Reda Snaiki1*, Rocio Lilen Segura2 
1 Department of Construction Engineering, École de Technologie Supérieure, Université du Québec, 
Montréal, Québec, Canada 
2 Department of Civil, Geological and Mining Engineering, Polytechnique Montréal, 2500 Chemin de 
Polytechnique, Montréal, QC H3T 0A3, Canada 
 
*Corresponding author. Email: reda.snaiki@etsmtl.ca  


In [ ]:
import numpy as np, faiss
from sentence_transformers import SentenceTransformer

# 1. Chunk each page into ~200-word pieces with overlap, keeping page numbers
chunks, meta = [], []
for page_no, page_text in enumerate(pages, start=1):
    words = page_text.split()
    for i in range(0, len(words), 150):
        piece = " ".join(words[i:i+200])
        if len(piece.split()) > 30:
            chunks.append(piece)
            meta.append({"page": page_no})

# 2. Embed the chunks
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
emb = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)

# 3. Build the FAISS index
index = faiss.IndexFlatIP(emb.shape[1])
index.add(np.array(emb, dtype="float32"))

print("Chunks:", len(chunks), "| Index size:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Chunks: 91 | Index size: 91


In [ ]:
def retrieve(question, k=4):
    q = embedder.encode([question], normalize_embeddings=True)
    scores, ids = index.search(np.array(q, dtype="float32"), k)
    return [(chunks[i], meta[i]["page"]) for i in ids[0]]

def rag_answer(question):
    hits = retrieve(question)
    context = "\n\n".join(f"[Page {p}] {c}" for c, p in hits)
    prompt = (
        "Answer the question using ONLY the context below. "
        "Cite the page like [Page 5] after each claim. "
        "If the context does not contain the answer, say: Not found in the paper.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    answer = ask(prompt, max_new_tokens=300)
    pages_used = sorted({p for _, p in hits})
    return answer, pages_used

answer, pages_used = rag_answer("What is the main contribution of this paper?")
print(answer)
print("\nSource pages retrieved:", pages_used)

The main contribution of this paper, as stated in [Page 24], is the development of a "multimodal RAG (Retrieval-Augmented Generator) framework for housing damage assessment." This framework focuses on collaborative optimization of image encoding and policy vector retrieval for housing damage assessment tasks.

Source pages retrieved: [33, 34]


In [ ]:
import re

# Find where the reference list starts (a line that is just "References")
ref_page = len(pages) + 1
for n, t in enumerate(pages, start=1):
    if n > len(pages) // 2 and any(l.strip().lower() in ("references", "reference") for l in t.splitlines()):
        ref_page = n
        break
print("Reference list starts on page:", ref_page)

def retrieve(question, k=4):
    q = embedder.encode([question], normalize_embeddings=True)
    scores, ids = index.search(np.array(q, dtype="float32"), 20)
    hits = [(chunks[i], meta[i]["page"]) for i in ids[0] if meta[i]["page"] < ref_page]
    return hits[:k]

def verify_citations(answer, pages_used):
    cited = {int(n) for n in re.findall(r"\[Page (\d+)\]", answer)}
    fake = sorted(cited - set(pages_used))
    return cited, fake

answer, pages_used = rag_answer("What is the main contribution of this paper?")
cited, fake = verify_citations(answer, pages_used)
print(answer)
print("\nRetrieved pages:", pages_used)
print("Cited pages:", sorted(cited))
print("UNSUPPORTED citations:", fake if fake else "none")

Reference list starts on page: 32
The main contribution of this paper is not explicitly stated on the provided pages [Page 24]. However, based on the context, the paper introduces a method called GraphRAG that focuses on connecting observed damage to recovery relationships and downstream consequences [Page 24]. This method also retains the capability to trace predictions to the corresponding evidence, engineering relationships, and provenance for outcomes with comparable performance [Page 24]. The paper also emphasizes the construction of a knowledge-graph that integrates case-specific observations, standardized engineering concepts, and recovery information, while preserving the origin of the evidence [Page 7].

Retrieved pages: [7, 9, 13, 24]
Cited pages: [7, 24]
UNSUPPORTED citations: none


In [ ]:
print(pages[23][:2500])

24 
 
major localized roof-structure damage is confirmed, the assessment changes to 𝑆1, 𝐻2, 
and 𝐹3, with major repair and a substantially longer downtime estimate. 
Structured BaselineRAG relies on states and recovery information associated with 
semantically similar scenarios, whereas GraphRAG explicitly connects the validated 
observations to supported engineering relationships. In particular, temporary roof 
protection is retained as a temporary measure rather than confirmation of the concealed 
roof condition. GraphRAG consequently distinguishes directly supported recovery 
actions from precautionary or conditional actions that depend on confirmation of 
unresolved conditions. The example therefore illustrates the added traceability and 
evidence control provided by graph-aligned reasoning, as well as its ability to preserve 
conditional assessment branches when the available observations do not establish the 
underlying damage condition. 
Aggregate performance was evaluated on th

In [ ]:
def check_claims(answer):
    sentences = re.split(r"(?<=[.!?])\s+", answer)
    rows = []
    for s in sentences:
        cited = [int(n) for n in re.findall(r"\[Page (\d+)\]", s)]
        if not cited:
            continue
        clean = re.sub(r"\[Page \d+\]", "", s).strip()
        s_emb = embedder.encode([clean], normalize_embeddings=True)[0]
        best = max(float(emb[i] @ s_emb) for i, m in enumerate(meta) if m["page"] in cited)
        rows.append((round(best, 2), cited, clean[:110]))
    return rows

for score, cited, text in check_claims(answer):
    print(score, cited, text)

0.05 [24] The main contribution of this paper is not explicitly stated on the provided pages .
0.83 [24] However, based on the context, the paper introduces a method called GraphRAG that focuses on connecting observ
0.56 [24] This method also retains the capability to trace predictions to the corresponding evidence, engineering relati
0.72 [7] The paper also emphasizes the construction of a knowledge-graph that integrates case-specific observations, st


In [ ]:
def report(question):
    ans, pgs = rag_answer(question)
    print("Q:", question)
    print(ans)
    print("Retrieved pages:", pgs)
    for score, cited, text in check_claims(ans):
        flag = "OK    " if score >= 0.6 else "REVIEW"
        print(f"  {flag} {score} {cited} {text}")
    print("=" * 60)

for q in [
    "What datasets or case study data does the paper use?",
    "What are the limitations of the proposed framework?",
    "What is the capital of France?",
]:
    report(q)

NameError: name 'rag_answer' is not defined

In [ ]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu pymupdf

import os, re, torch, faiss, pymupdf, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# --- Granite ---
model_id = "ibm-granite/granite-3.3-2b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="cuda")

def ask(question, max_new_tokens=300):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# --- Read paper ---
path = next(p for p in ["/content/testpaper.pdf", "/testpaper.pdf"] if os.path.exists(p))
pages = [pg.get_text() for pg in pymupdf.open(path)]

# --- Chunk, embed, index ---
chunks, meta = [], []
for page_no, page_text in enumerate(pages, start=1):
    words = page_text.split()
    for i in range(0, len(words), 150):
        piece = " ".join(words[i:i+200])
        if len(piece.split()) > 30:
            chunks.append(piece); meta.append({"page": page_no})
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
emb = embedder.encode(chunks, normalize_embeddings=True)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(np.array(emb, dtype="float32"))

# --- Skip the reference list ---
ref_page = len(pages) + 1
for n, t in enumerate(pages, start=1):
    if n > len(pages) // 2 and any(l.strip().lower() in ("references", "reference") for l in t.splitlines()):
        ref_page = n
        break

def retrieve(question, k=4):
    q = embedder.encode([question], normalize_embeddings=True)
    _, ids = index.search(np.array(q, dtype="float32"), 20)
    return [(chunks[i], meta[i]["page"]) for i in ids[0] if meta[i]["page"] < ref_page][:k]

def rag_answer(question):
    hits = retrieve(question)
    context = "\n\n".join(f"[Page {p}] {c}" for c, p in hits)
    prompt = ("Answer the question using ONLY the context below. "
              "Cite the page like [Page 5] after each claim. "
              "If the context does not contain the answer, say: Not found in the paper.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}")
    return ask(prompt), sorted({p for _, p in hits})

def check_claims(answer):
    rows = []
    for s in re.split(r"(?<=[.!?])\s+", answer):
        cited = [int(n) for n in re.findall(r"\[Page (\d+)\]", s)]
        if not cited:
            continue
        clean = re.sub(r"\[Page \d+\]", "", s).strip()
        s_emb = embedder.encode([clean], normalize_embeddings=True)[0]
        best = max(float(emb[i] @ s_emb) for i, m in enumerate(meta) if m["page"] in cited)
        rows.append((round(best, 2), cited, clean[:110]))
    return rows

def report(question):
    ans, pgs = rag_answer(question)
    print("Q:", question); print(ans); print("Retrieved pages:", pgs)
    for score, cited, text in check_claims(ans):
        print(f"  {'OK    ' if score >= 0.6 else 'REVIEW'} {score} {cited} {text}")
    print("=" * 60)

for q in ["What datasets or case study data does the paper use?",
          "What are the limitations of the proposed framework?",
          "What is the capital of France?"]:
    report(q)

config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.93k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.48M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/29.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

StopIteration: 

In [ ]:
import glob

# --- Read paper (finds your PDF automatically) ---
found = glob.glob("/content/*.pdf") + glob.glob("/*.pdf")
print("PDF found:", found)
path = found[0]
pages = [pg.get_text() for pg in pymupdf.open(path)]

# --- Chunk, embed, index ---
chunks, meta = [], []
for page_no, page_text in enumerate(pages, start=1):
    words = page_text.split()
    for i in range(0, len(words), 150):
        piece = " ".join(words[i:i+200])
        if len(piece.split()) > 30:
            chunks.append(piece); meta.append({"page": page_no})
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")
emb = embedder.encode(chunks, normalize_embeddings=True)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(np.array(emb, dtype="float32"))

# --- Skip the reference list ---
ref_page = len(pages) + 1
for n, t in enumerate(pages, start=1):
    if n > len(pages) // 2 and any(l.strip().lower() in ("references", "reference") for l in t.splitlines()):
        ref_page = n
        break

def retrieve(question, k=4):
    q = embedder.encode([question], normalize_embeddings=True)
    _, ids = index.search(np.array(q, dtype="float32"), 20)
    return [(chunks[i], meta[i]["page"]) for i in ids[0] if meta[i]["page"] < ref_page][:k]

def rag_answer(question):
    hits = retrieve(question)
    context = "\n\n".join(f"[Page {p}] {c}" for c, p in hits)
    prompt = ("Answer the question using ONLY the context below. "
              "Cite the page like [Page 5] after each claim. "
              "If the context does not contain the answer, say: Not found in the paper.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}")
    return ask(prompt), sorted({p for _, p in hits})

def check_claims(answer):
    rows = []
    for s in re.split(r"(?<=[.!?])\s+", answer):
        cited = [int(n) for n in re.findall(r"\[Page (\d+)\]", s)]
        if not cited:
            continue
        clean = re.sub(r"\[Page \d+\]", "", s).strip()
        s_emb = embedder.encode([clean], normalize_embeddings=True)[0]
        best = max(float(emb[i] @ s_emb) for i, m in enumerate(meta) if m["page"] in cited)
        rows.append((round(best, 2), cited, clean[:110]))
    return rows

def report(question):
    ans, pgs = rag_answer(question)
    print("Q:", question); print(ans); print("Retrieved pages:", pgs)
    for score, cited, text in check_claims(ans):
        print(f"  {'OK    ' if score >= 0.6 else 'REVIEW'} {score} {cited} {text}")
    print("=" * 60)

for q in ["What datasets or case study data does the paper use?",
          "What are the limitations of the proposed framework?",
          "What is the capital of France?"]:
    report(q)

PDF found: ['/content/testpaper.pdf']


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Q: What datasets or case study data does the paper use?
Not found in the paper. The context provided does not mention any datasets or case study data used in the paper.
Retrieved pages: [5, 7, 11, 14]
Q: What are the limitations of the proposed framework?
The limitations of the proposed framework, as indicated in the context, are:

1. Removing engineering context produces the largest macro-F1 reductions for safety and functionality, with functionality macro-F1 decreasing from 0.824 to 0.434 [Page 26].
2. Removing evidence safeguards reduces functionality macro-F1 to 0.586 and produces the largest habitability macro-F1 reduction, from 0.984 to 0.602 [Page 26].
3. Removing graph-path traversal also reduces performance, with functionality macro-F1 decreasing to 0.751 [Page 26].

These limitations suggest that the framework's performance, particularly in terms of safety, functionality, and habitability, deteriorates when engineering context, evidence safeguards, and graph-path traversal ar

In [ ]:
for kw in ["limitation", "dataset", "data set", "case study"]:
    found = [n for n, t in enumerate(pages, start=1) if kw in t.lower()]
    print(f"{kw!r} appears on pages: {found}")

'limitation' appears on pages: [4, 30]
'dataset' appears on pages: [2, 15, 33]
'data set' appears on pages: []
'case study' appears on pages: [4, 15, 16, 21, 24, 29, 31]


In [ ]:
def retrieve(question, k=8):
    q = embedder.encode([question], normalize_embeddings=True)
    scores, ids = index.search(np.array(q, dtype="float32"), 40)
    words = [w[:6] for w in re.findall(r"[a-z]+", question.lower()) if len(w) > 4]
    ranked = []
    for s, i in zip(scores[0], ids[0]):
        if meta[i]["page"] >= ref_page:
            continue
        bonus = 0.05 * sum(w in chunks[i].lower() for w in words)
        ranked.append((s + bonus, i))
    ranked.sort(reverse=True)
    return [(chunks[i], meta[i]["page"]) for _, i in ranked[:k]]

for q in ["What datasets or case study data does the paper use?",
          "What are the limitations of the proposed framework?"]:
    report(q)

Q: What datasets or case study data does the paper use?
The paper uses post-event reconnaissance reports and associated datasets produced by the StEER [33–37], as well as relevant research publications [38–41] for the tornado case study [Page 15]. These sources provide building-level photographs, damage observations, construction information, and reported assessment information. Image-based scenarios were obtained from StEER datasets, while document-derived scenarios were extracted from report sections and research publications using building descriptions, figure captions, and associated assessment information. The case study focuses on tornado-affected low-rise residential buildings, defined as buildings up to three stories above grade [Page 15].
Retrieved pages: [5, 7, 10, 14, 15, 18]
  OK     0.68 [15] The paper uses post-event reconnaissance reports and associated datasets produced by the StEER [33–37], as wel
  OK     0.71 [15] The case study focuses on tornado-affected low-rise r

In [ ]:
for pno in [30, 4]:
    t = pages[pno - 1]
    i = t.lower().find("limitation")
    print(f"--- Page {pno} ---")
    print(t[max(0, i - 300): i + 900])
    print()

--- Page 30 ---
y 
time. However, these downtime estimates remain engineering-model outputs based on 
the prescribed recovery-action library, duration distributions, and dependency 
relationships; they should not be interpreted as empirically validated predictions of 
actual building-level recovery times. 
Several limitations remain. The current implementation is restricted to tornado-affected 
low-rise residential buildings and primarily exterior post-event evidence, limiting direct 
generalization to other building types, occupancies, and wind hazards. The quality of the 
resulting inference also depends on the completeness of the reconnaissance evidence and 


--- Page 4 ---
t, evidence safeguards, and graph-path traversal. The 
case study therefore evaluates both predictive agreement with the operational reference 
states and the added value of traceable, recovery-oriented engineering reasoning. 
2. Methodology 
2.1 Framework overview 
The framework addresses a common limitation of

In [ ]:
def check_claims(answer, pages_used):
    rows = []
    for s in re.split(r"(?<=[.!?])\s+", answer):
        cited = [int(n) for n in re.findall(r"\[Page (\d+)\]", s)]
        if not cited:
            continue
        clean = re.sub(r"\[Page \d+\]", "", s).strip()
        s_emb = embedder.encode([clean], normalize_embeddings=True)[0]
        sims = {}
        for i, m in enumerate(meta):
            if m["page"] in pages_used:
                sims[m["page"]] = max(sims.get(m["page"], 0), float(emb[i] @ s_emb))
        best = max(sims, key=sims.get)
        cited_score = max(sims.get(c, 0) for c in cited)
        rows.append((round(cited_score, 2), cited, best, round(sims[best], 2), clean[:100]))
    return rows

def report(question):
    ans, pgs = rag_answer(question)
    print("Q:", question); print(ans); print("Retrieved pages:", pgs)
    for score, cited, best, bscore, text in check_claims(ans, pgs):
        flag = "OK    " if score >= 0.6 else "REVIEW"
        if best not in cited and bscore - score > 0.1:
            flag = "WRONG PAGE?"
        print(f"  {flag} cited {cited} score {score} | best page {best} ({bscore}) | {text}")
    print("=" * 60)

report("What are the limitations of the proposed framework?")

Q: What are the limitations of the proposed framework?
The limitations of the proposed framework, as mentioned in the context, include being restricted to tornado-affected low-rise residential buildings and primarily focusing on severity, extent, structural scope, consequences, and recovery relationships [Page 26]. Additionally, the current implementation does not account for practical post-event information supplied through images, reconnaissance text, or engineer-generated descriptions rather than through a single standardized modality [Page 26]. The recovery component, which links supported physical conditions to intervention requirements and downtime, is also limited as recovery actions are inferred from accepted damage–consequence–action relationships rather than transferred wholesale from a retrieved scenario [Page 26]. Furthermore, the framework's recovery estimates should not be interpreted as empirically validated predictions of actual building-level recovery times [Page 26].


In [ ]:
for q in ["What is the main contribution of this paper?",
          "What datasets or case study data does the paper use?",
          "What are the limitations of the proposed framework?",
          "What is the capital of France?"]:
    report(q)

Q: What is the main contribution of this paper?
The main contribution of this paper is the development of a knowledge-graph framework that integrates case-specific observations, standardized engineering concepts, and recovery information, while preserving the evidence on which these concepts are based [Page 7]. This framework allows for the inference of building conditions, functionality, and recovery actions, with the spatial and structural scope of observed damage being maintained [Page 13]. The graph's utility depends on encoding meaningful engineering structure and having explicit controls for evidence transfer across cases [Page 29]. The paper also highlights the importance of controlled fallback evidence and retaining only one representative from a given source family at the active evidence level [Page 12]. Furthermore, the framework enables tracing predictions to the corresponding evidence, engineering relationships, and provenance, providing additional capabilities for outcomes